IMPORTS and GLOBALS

In [ ]:
import torch
import torch.nn as nn
import math
# import torch.optim as optim
# from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from tqdm.notebook import tqdm
from livelossplot import PlotLosses 
import numpy as np
# from transformers import AutoModelForCausalLM, AutoTokenizer, LlamaForCausalLM
import os
from helper import get_model_path, load_model_and_tokenizer, get_embedding_matrix, get_tokens, create_one_hot_and_embeddings, get_nonascii_toks
import torch.nn.functional as F
device: str = "cuda:0"
num_steps: int = 200
num_tokens: int = 300 
step_size: float = 0.1

seed: int = 42
load_csv_data = True
# verbose = True
# early_stopping = True

In [ ]:
# Use Config and Samuel's directory to load different kinds of models
# MODEL PATHS    0           1       2       3           4
model_options = ["Falcon", "Llama2", "MPT", "Vicuna", "Mistral"] 
model_name = model_options[1]
model_path: str = get_model_path(model_name=model_name)
print(model_path)

In [ ]:
if seed is not None:
        torch.manual_seed(seed)

model, tokenizer = load_model_and_tokenizer(
        model_path, low_cpu_mem_usage=True, use_cache=False, device=device
    )

In [ ]:
non_ascii_toks = get_nonascii_toks(tokenizer, device=device).tolist()
# Assuming device is already defined
non_ascii_toks_tensor = torch.tensor(non_ascii_toks).to(device=device)


def get_masked_one_hot_adv(one_hot_adv):
    # Mask out all the non-ascii tokens and then return
    # Step 1: Create a tensor with all the non_ascii tokens
    top_token_ids_tensor_2d = non_ascii_toks_tensor.unsqueeze(0).repeat(20, 1)
    
    # Step 2: Create a mask with the same shape as one_hot_adv, initialized to zero
    mask = torch.ones_like(one_hot_adv, dtype=torch.float16)

    # Step 3: Use token_ids_init to set the corresponding indices in the mask to 1
    # We use gather and scatter operations for this
    mask.scatter_(1, top_token_ids_tensor_2d, 0.0)

    # Step 4: Apply the mask to one_hot_adv
    masked_one_hot_adv = one_hot_adv * mask
    
    return masked_one_hot_adv

In [ ]:
def calc_loss(model, embeddings_user, embeddings_adv, embeddings_target, targets):
    full_embeddings = torch.hstack([embeddings_user, embeddings_adv, embeddings_target])
    outputs = model(inputs_embeds=full_embeddings, output_hidden_states=True, use_cache=False)
    # logits = model(inputs_embeds=full_embeddings).logits
    loss_slice_start = len(embeddings_user[0]) + len(embeddings_adv[0])
    loss = nn.CrossEntropyLoss()(outputs.logits[0, loss_slice_start - 1 : -1, :], targets)
    return loss, outputs
    # return loss, outputs.logits[:, loss_slice_start-1:, :]

In [ ]:
class EGDwithAdamOptimizer(torch.optim.Optimizer):
    def __init__(self, params, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-4, grad_clip_val=1.0):
        defaults = dict(lr=lr, beta1=beta1, beta2=beta2, eps=eps, grad_clip_val=grad_clip_val)
        super(EGDwithAdamOptimizer, self).__init__(params, defaults)

        # Initialize state variables for each parameter group
        for group in self.param_groups:
            for param in group['params']:
                self.state[param] = {
                    'm': torch.zeros_like(param),  # First moment vector (momentum)
                    'v': torch.zeros_like(param),  # Second moment vector (variance)
                    't': 0                         # Time step
                }

    def step(self, closure=None):
        loss = None
        if closure is not None:
            loss = closure()

        for group in self.param_groups:
            lr = group['lr']
            beta1 = group['beta1']
            beta2 = group['beta2']
            eps = group['eps']
            # grad_clip_val = group['grad_clip_val']

            for param in group['params']:
                if param.grad is None:
                    continue

                grad = param.grad
                state = self.state[param]

                # Retrieve state variables
                m = state['m']
                v = state['v']
                t = state['t']

                # Increment time step
                t += 1
                state['t'] = t

                # # Clip gradients to avoid exploding updates; Redundant step
                # grad = torch.clamp(grad, min=-grad_clip_val, max=grad_clip_val)

                # Update biased first moment estimate (m) and second moment estimate (v)     
                m = beta1 * m + (1 - beta1) * grad
                v = beta2 * v + (1 - beta2) * torch.pow(grad, 2) 
                # v = beta2 * v + (1 - beta2) * (grad ** 2)
                state['m'] = m
                state['v'] = v

                # Bias correction
                m_hat = m / (1 - math.pow(beta1, t)) # m_hat = m / (1 - beta1 ** t)
                v_hat = v / (1 - math.pow(beta2, t)) # v_hat = v / (1 - beta2 ** t)

                # Adam-like modified gradient
                modified_grad = m_hat / (torch.sqrt(v_hat) + eps)

                with torch.no_grad():
                    # Exponentiated Gradient Descent update with Adam-like modifications
                    param.mul_(torch.exp(-lr * modified_grad))

                    # Clamp the parameter values to avoid extreme values
                    param.clamp_(min=1e-12, max=1e12)

                    # Normalize each row to sum up to 1, considering possible tensor shapes
                    if param.dim() > 1:
                        row_sums = param.sum(dim=1, keepdim=True) + 1e-10  # Add epsilon to avoid division by zero
                        param.div_(row_sums)
                    else:
                        # Handle 1D tensors or other cases if applicable
                        param.div_(param.sum() + 1e-10)

        return loss


BEGIN ATTACK HERE

In [ ]:
import csv
embed_weights = get_embedding_matrix(model)
# Datset options:       0           1               2                   3
dataset_options = ['AdvBench', 'HarmBench', 'JailbreakBench', 'MaliciousInstruct']
dataset_name = dataset_options[0]
dataset_path: str = f"./data/{dataset_name}/harmful_behaviors.csv"
if load_csv_data:
    reader = csv.reader(open(dataset_path, 'r'))
    next(reader)
else:
    print("Error!")
harmful_behaviors = list(reader)[0:5]
# We can hard-code this safe_target string; probably
# safe_target: str = "I cannot fulfill your request. I'm just an AI."
print(len(harmful_behaviors))
for each_behavior in harmful_behaviors:
    print(each_behavior)

In [ ]:
print(f'model_name: {model_name}')
print(f'dataset_name: {dataset_name}')
print(f'vocabulary size: {tokenizer.vocab_size}')

In [ ]:
# # Config-level (works even before loading weights fully)
# vocab_size   = model.config.vocab_size
# hidden_size  = model.config.hidden_size
# tied         = getattr(model.config, "tie_word_embeddings", None)

# print('vocab_size: ',vocab_size, 'hidden_size:', hidden_size, 'tied?',tied)  # e.g., 32000, 4096, False

# # Parameter-level (reads actual tensor shapes)
# emb_shape    = model.get_input_embeddings().weight.shape           # [vocab_size, hidden_size]
# lm_head_shape= model.get_output_embeddings().weight.shape          # [hidden_size, vocab_size]

# print(emb_shape, lm_head_shape)

In [ ]:
# # Pick a token to test
# tokens = get_tokens("apple", tokenizer=tokenizer, device=device)
# one_hot_inputs, _ = create_one_hot_and_embeddings(tokens, embed_weights, device=device)
# # token = "apple"
# # token_id = tokenizer.convert_tokens_to_ids(token)

# # === Input embedding ===
# embed_matrix = model.get_input_embeddings().weight           # [vocab_size, hidden_size]
# input_vec = one_hot_inputs@embed_matrix                           # [hidden_size]

# # === Output embedding ===
# # Note: For LLaMA, lm_head.weight also has shape [vocab_size, hidden_size],
# # and logits are computed as hidden @ lm_head.weight.T.
# lm_head_weight = model.get_output_embeddings().weight         # [vocab_size, hidden_size]
# output_vec = one_hot_inputs@lm_head_weight                       # [hidden_size]

# # === Cosine similarity ===
# cos_sim = F.cosine_similarity(input_vec, output_vec, dim=1)

# print(f"Token: {tokens}")
# print(f"Input embedding shape:  {input_vec.shape}")
# print(f"Output embedding shape: {output_vec.shape}")
# print(f"Cosine similarity:      {cos_sim}")
# # print(f"Cosine similarity:      {cos_sim.item():.4f}")

# # === Optional: check if they are tied ===
# same_params = embed_matrix.data_ptr() == lm_head_weight.data_ptr()
# print("Are embeddings tied?", same_params)

In [ ]:
def get_cosine_sim(model, outputs, targets, embeddings_user, embeddings_adv):
    """
    Compute cosine similarity between RMSNorm(hidden_state) and LM head vectors for target tokens.
    """
    # === Get hidden states from last transformer layer ===
    hidden_states = outputs.hidden_states[-1]  # shape: [1, seq_len, hidden_size]

    # === Get model's final RMSNorm ===
    rms_norm = model.model.norm  # Hugging Face LLaMA models store this here

    # === Normalize the hidden states ===
    normed_hidden = rms_norm(hidden_states)  # shape: [1, seq_len, hidden_size]

    # === Figure out the slice of interest ===
    loss_slice_start = len(embeddings_user[0]) + len(embeddings_adv[0])  # like your loss slicing
    normed_hidden_slice = normed_hidden[0, loss_slice_start - 1 : -1, :]  # [T, H]

    # === Get LM head weights ===
    lm_head_weights = model.lm_head.weight  # shape: [vocab_size, hidden_size]

    # === Get unembedding vectors for the correct target tokens ===
    target_vectors = lm_head_weights[targets]  # shape: [T, H]

    # === Normalize both for cosine similarity ===
    normed_hidden_vecs = F.normalize(normed_hidden_slice, dim=-1)  # [T, H]
    normed_target_vecs = F.normalize(target_vectors, dim=-1)       # [T, H]

    # === Cosine similarity and distance ===
    cosine_sim = torch.sum(normed_hidden_vecs * normed_target_vecs, dim=-1)  # [T]
    # cosine_dist = 1.0 - cosine_sim  # [T]
    return  cosine_sim
    # return cosine_dist, cosine_sim


In [ ]:
# Update the one_hot encodings using Gradient Descent.
from math import *
import json 
import pandas as pd
import torch.nn.functional as F

# Fetch the Model's weights
embed_weights = get_embedding_matrix(model)
# reader = [[user_prompt, target]]
optimization_results = []
# Set your regularization parameter
# initial_coefficient = 1e-5
# final_coefficient = 1e-3
# === Regularization Scheduling Setup ===
# Entropy regularization: decay (strong early → off late)
initial_entropy_coeff = 1e-4     # High to encourage soft exploration
final_entropy_coeff   = 1e-8     # Turns off by end

# Peakiness regularization: growth (off early → strong late)
initial_peakiness_coeff = 1e-8    # No pressure early
final_peakiness_coeff   = 1e-3   # Encourages discreteness at end

initial_lambda = 0.01
final_lambda = 0.3

# Create the following directories if they do NOT exist
directories = [f'./CSV_Files/Single-Prompt/{model_name}/{dataset_name}',       # 0
               f'./JSON_Files/Single-Prompt//{model_name}/{dataset_name}']      # 1

for directory in directories:
    # Create the directory if it doesn't exist
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Directory '{directory}' created.")
    else:
        print(f"Directory '{directory}' already exists.")

    
# JSON file to dump the model generated outputs 
json_file_path = directories[1]+"/output_("+str(len(harmful_behaviors))+"_behaviors)("+str(num_steps)+"_steps).json"

for row in tqdm(harmful_behaviors, desc="Optimizing prompts"):
    iteration_result = {}
    user_prompt, target = row
    print(user_prompt)
    # target = unsafe_target
    user_prompt_tokens = get_tokens(user_prompt, tokenizer=tokenizer, device=device)    
    target_tokens = get_tokens(target, tokenizer=tokenizer, device=device)[1:]

    # Initialize one_hot encodings and embedding vectors for the user prompts and targets 
    _, embeddings_user = create_one_hot_and_embeddings(user_prompt_tokens, embed_weights, device=device)
    _, embeddings_target = create_one_hot_and_embeddings(target_tokens, embed_weights, device=device)
    best_disc_loss = np.inf
    best_loss_at_epoch = 0
    # # Initialize the adversarial one_hot encodings using ONLY ASCII tokens
    # one_hot_adv = get_only_ASCII_one_hot_adv()
    # # Do NOT Initialize the adversarial one_hot encodings using all '!' tokens
    # one_hot_adv = get_all_exclamations_one_hot_adv(embed_weights=embed_weights)
    # You can also initialize one_hot_adv Randomly.
    # one_hot_adv = F.softmax(torch.rand(20, tokenizer.vocab_size, dtype=torch.float16).to(device=device), dim=1).to(embed_weights.dtype)
    one_hot_adv = F.softmax(torch.rand(20, embed_weights.shape[0], dtype=torch.float16).to(device=device), dim=1).to(embed_weights.dtype)
    one_hot_adv.requires_grad_() 
    # Initialize the optimizer
    optimizer = EGDwithAdamOptimizer([one_hot_adv], lr=step_size)
    # Initialize the learning_rate scheduler
    scheduler_cycle = 0
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=50)
    # Specify the filename
    last_three_words = lambda text: '_'.join(text.split()[-3:])
    first_three_words = lambda text: '_'.join(text.split()[0:3])
    csv_filename = directories[0]+'/stats_('+first_three_words(user_prompt)+".."+last_three_words(user_prompt)+')('+str(num_steps)+'_steps).csv'        
    # Generate column names
    column_names = ['epoch', 'cycle', 'learning_rate', 'entropy_term', 
                    'kl_divergence_term', 
                    'continuous_loss','discrete_loss'] 
    # Adding 'max_1' to 'max_20' column names using a loop
    for i in range(1, 21):
        column_names.append(f'max_{i}')
    for i in range(1, 21):
        column_names.append(f'token_id_{i}')
    # Create an empty DataFrame
    df = pd.DataFrame(columns=column_names)

    # PHASE_2: bool=False
    BREAK_IT: bool=False

    for epoch_no in tqdm(range(num_steps)):
        optimizer.zero_grad()

        embeddings_adv = (one_hot_adv @ embed_weights).unsqueeze(0)
        cross_entropy_loss, model_outputs = calc_loss(model, embeddings_user, embeddings_adv, embeddings_target, target_tokens)
        continuous_loss = cross_entropy_loss.detach().cpu().item()
        # Break the loop if loss becomes "NaN"
        if math.isnan(continuous_loss):
            print("The value is NaN.")
            BREAK_IT=True # Go to the Next Behavior

        # Current epoch progress
        progress = epoch_no / (num_steps - 1 + 1e-8)  # avoid divide-by-zero
        # For entropy: decay
        entropy_reg_coeff = initial_entropy_coeff * (final_entropy_coeff / initial_entropy_coeff + 1e-12) ** progress

        # For peakiness: growth
        peakiness_reg_coeff = (initial_peakiness_coeff + 1e-12) * (final_peakiness_coeff / (initial_peakiness_coeff + 1e-12)) ** progress
        # reg_coefficient = initial_coefficient * (final_coefficient / initial_coefficient) ** (epoch_no / (num_steps - 1))           
        ################# Usual entropy #################
        # # Calculate H(X) = − ∑_{i=1}^{L} ∑_{j=1}^{|T|} X_{ij} (log X_{ij} −1)
        # Adding a small epsilon to avoid log(0) which is undefined
        eps = 1e-12
        one_hot_adv_float32 = one_hot_adv.to(dtype=torch.float32)
        log_one_hot = torch.log(one_hot_adv_float32 + eps)
        # Compute the modified entropy
        entropy_row_col = -one_hot_adv * (log_one_hot - 1)
        entropy_term = entropy_row_col.sum()
        entropy_term *= entropy_reg_coeff
        # Calculate the annealed coefficient, 𝜖
        # Use Exponential Scheduling
        # Take the Negative log of the highest probability in each row
        # peakiness_term = -torch.log(torch.max(one_hot_adv, dim=1).values + eps).sum()
        max_per_row = torch.max(one_hot_adv_float32, dim=1).values      # [L]
        neg_log_max = -torch.log(max_per_row + eps)             # [L]
        peakiness_term = neg_log_max.sum()                      # scalar
        peakiness_term *=peakiness_reg_coeff

        # Experimental Starts
        # === Compute MSE loss / Cosine Distance ===
        lambda_reg = initial_lambda * (final_lambda / initial_lambda) ** (epoch_no / (num_steps - 1))

        cosine_sim = get_cosine_sim(model, model_outputs, target_tokens, embeddings_user, embeddings_adv)
        # cosine_dist = 1.0 - cosine_sim
        # print("cosine similarity:", [float(x) for x in cosine_sim], "Mean:", cosine_sim.mean().item())
        cosine_reg_term = -cosine_sim.mean()  # encourage alignment by maximizing similarity
        cosine_loss = lambda_reg * cosine_reg_term        
        # Regularized loss: F(X) - H(X) + KL(X|~X) + reg_mse
        regularized_loss = cross_entropy_loss - entropy_term + peakiness_term + cosine_loss 

        # Experiment Ends
        
        # Backpropagate the regularized loss
        regularized_loss.backward()

        # Get the scheduler's state to get learning_rate
        scheduler_state = scheduler.state_dict()

        # Clip the gradients(before update)
        torch.nn.utils.clip_grad_norm_([one_hot_adv], max_norm=1.0)

        # update parameters
        optimizer.step()
        # update scheduler
        scheduler.step(continuous_loss)
        
        # Discretization Starts
        masked_one_hot_adv = get_masked_one_hot_adv(one_hot_adv)
        # Use this masked_one_hot_adv ONLY for discretization
        max_values = torch.max(masked_one_hot_adv, dim=1)
        adv_token_ids = max_values.indices
        # adv_token_ids = top_k_sampling(one_hot_adv, k=10).squeeze(dim=1)
        # tau = initial_tau * (final_tau / initial_tau) ** (epoch_no / (num_steps - 1))
        # one_hot_discrete = F.gumbel_softmax(one_hot_adv, tau=tau, hard=True)
        # adv_token_ids = one_hot_discrete.argmax(dim=1)
        one_hot_discrete = torch.zeros(
            adv_token_ids.shape[0], embed_weights.shape[0], device=device, dtype=embed_weights.dtype
        )
        one_hot_discrete.scatter_(
            1,
            adv_token_ids.unsqueeze(1),
            torch.ones(masked_one_hot_adv.shape[0], 1, device=device, dtype=embed_weights.dtype),
        )
        # Use one_hot_discrete to print Tokens
        # What other techniques Can we use here to discretize the one_hot encodings?
        # Use discrete tokens to calculate loss
        with torch.no_grad():
            embeddings_adv_discrete = (one_hot_discrete @ embed_weights).unsqueeze(0)
            disc_loss, _ = calc_loss(model, embeddings_user, embeddings_adv_discrete, embeddings_target, target_tokens)
        # If loss improves, save it as x_best
        discrete_loss =  disc_loss.detach().cpu().item()
        # cur_loss_list.append(cur_loss)
        if discrete_loss < best_disc_loss:
            # print(f"########## {discrete_loss} #########")
            best_disc_loss = discrete_loss
            best_loss_at_epoch = epoch_no
            effective_adv_embeddings = embeddings_adv_discrete
            effective_adv_one_hot = one_hot_discrete
        else :
            pass
        # # Discretization Ends
        max_values = torch.max(masked_one_hot_adv, dim=1)
        adv_token_ids = max_values.indices
        # Dump Ouput values to a CSV file
        # Convert max_values to a NumPy array
        max_values_array = max_values.values.detach().cpu().numpy()
        token_ids_array = adv_token_ids.detach().cpu().numpy()
        # Get the scheduler's state and learning_rate
        # Access the current learning rate directly from the optimizer's parameter group
        scheduler_lr = optimizer.param_groups[0]['lr']  # Get the learning rate of the first parameter group
        # scheduler_lr = scheduler_state['_last_lr'][0]
        # Create the Initial array       
        prepend_array = np.array([epoch_no, scheduler_cycle, scheduler_lr,
                                  entropy_term.detach().cpu().item(), 
                                  peakiness_term.detach().cpu().item(),
                                  continuous_loss, discrete_loss]) 
        
        # Concatenate the arrays
        row = np.concatenate((prepend_array, max_values_array, token_ids_array))
        new_row = pd.Series(row, index=df.columns)
        df = pd.concat([df, new_row.to_frame().T], ignore_index=True)
        # Save log data to CSV file periodically
        if epoch_no % 10 == 0:
            df.to_csv(csv_filename, index=False)
        # Something went wrong; so break it and go to the next behavior.
        if BREAK_IT==True:
            break
    # End of optimizations        
    # Write to CSV file
    df.to_csv(csv_filename, index=False)
    # Token and Output Generation Step
    # inputs_token_ids = one_hot_inputs.argmax(dim=1)
    adv_token_ids = effective_adv_one_hot.argmax(dim=1)
    adv_suffix_list = str(adv_token_ids.cpu().numpy().tolist())
    adv_suffix_string = tokenizer.decode(adv_token_ids.cpu().numpy())
    final_string_ids = torch.hstack([user_prompt_tokens, adv_token_ids])
    outputs = []
    generation_loop = range(1)
    import warnings
    # Suppress specific warnings
    warnings.filterwarnings("ignore", message=".*`do_sample` is set to `False`. However, `temperature` is set to.*")
    warnings.filterwarnings("ignore", message=".*`do_sample` is set to `False`. However, `top_p` is set to.*")
    
    ######## Discrete tokens for Output Generation ########
    for loop in tqdm(generation_loop, desc="Generating outputs", leave=False):
        generated_output = model.generate(final_string_ids.unsqueeze(0), 
                                          max_length=num_tokens, 
                                          pad_token_id=tokenizer.pad_token_id,
                                          do_sample=False)
        generated_output_string = tokenizer.decode(generated_output[0][:].cpu().numpy(), skip_special_tokens=True).strip()
        outputs.append(generated_output_string)
    
    iteration_result = {
        "harmful-behaviour": user_prompt,
        "target": target,
        "suffix_token_ids": adv_suffix_list,
        "suffix_string": adv_suffix_string,
        "epoch_no": epoch_no,
        "continuous_loss": continuous_loss,
        "discrete_loss": discrete_loss,
        "best_disc_loss":best_disc_loss,
        "epch_at_best_disc_loss": best_loss_at_epoch,
        "outputs": outputs
    }
    optimization_results.append(iteration_result)

    # Update the JSON File
    with open(json_file_path, "w") as f:
        json.dump(
            optimization_results,
            f,
            indent=4,  # Keep structure readable
            ensure_ascii=False  # Do this to Handle non-ASCII chars properly; If there is any
        )

    ##########################################################################################
    ## Create and Update a JSONL file (required for running the parse_results.ipynb script) ##
    output_file = directories[1]+"/output_"+model_name+"("+str(len(harmful_behaviors))+"_behaviors)("+str(num_steps)+"_steps).jsonl"
    from behavior import Behavior
    behavior = Behavior(user_prompt, adv_suffix_string, generated_output_string, "", "")
    
    with open(output_file, 'a') as f:
        f.write(json.dumps(behavior.to_dict()) + '\n')
    f.close()  
    ## Create and Update a JSONL file (required for running the parse_results.ipynb script) ##
    ########################################################################################## s

## Additional Codes

In [ ]:
# # Update the one_hot encodings using Gradient Descent.
# from math import *
# import json 
# import pandas as pd
# import torch.nn.functional as F

# # Fetch the Model's weights
# embed_weights = get_embedding_matrix(model)
# # reader = [[user_prompt, target]]
# optimization_results = []
# # Set your regularization parameter
# initial_coefficient = 1e-5
# final_coefficient = 1e-3
# initial_lambda = 0.01
# final_lambda = 0.2

# # Create the following directories if they do NOT exist
# directories = [f'./CSV_Files/Single-Prompt/emb_target_first(cosine)/{model_name}/{dataset_name}',       # 0
#                f'./JSON_Files/Single-Prompt/emb_target_first(cosine)/{model_name}/{dataset_name}']      # 1

# for directory in directories:
#     # Create the directory if it doesn't exist
#     if not os.path.exists(directory):
#         os.makedirs(directory)
#         print(f"Directory '{directory}' created.")
#     else:
#         print(f"Directory '{directory}' already exists.")

    
# # JSON file to dump the model generated outputs 
# json_file_path = directories[1]+"/output_("+str(len(harmful_behaviors))+"_behaviors)("+str(num_steps)+"_steps).json"

# for row in tqdm(harmful_behaviors, desc="Optimizing prompts"):
#     iteration_result = {}
#     user_prompt, target = row
#     print(user_prompt)
#     # target = unsafe_target
#     user_prompt_tokens = get_tokens(user_prompt, tokenizer=tokenizer, device=device)    
#     target_tokens = get_tokens(target, tokenizer=tokenizer, device=device)[1:]
#     plotlosses = PlotLosses()
#     plotlosses = PlotLosses(groups={'loss': ['continuous_loss', 'discrete_loss']})    
#     target_tokens = get_tokens(target, tokenizer=tokenizer, device=device)[1:]

#     # Initialize one_hot encodings and embedding vectors for the user prompts and targets 
#     one_hot_inputs, embeddings_user = create_one_hot_and_embeddings(user_prompt_tokens, embed_weights, device=device)
#     one_hot_target, embeddings_target = create_one_hot_and_embeddings(target_tokens, embed_weights, device=device)
#     best_disc_loss = np.inf
#     best_loss_at_epoch = 0
#     # # Initialize the adversarial one_hot encodings using ONLY ASCII tokens
#     # one_hot_adv = get_only_ASCII_one_hot_adv()
#     # # Do NOT Initialize the adversarial one_hot encodings using all '!' tokens
#     # one_hot_adv = get_all_exclamations_one_hot_adv(embed_weights=embed_weights)
#     # You can also initialize one_hot_adv Randomly.
#     # one_hot_adv = F.softmax(torch.rand(20, tokenizer.vocab_size, dtype=torch.float16).to(device=device), dim=1).to(embed_weights.dtype)
#     one_hot_adv = F.softmax(torch.rand(20, embed_weights.shape[0], dtype=torch.float16).to(device=device), dim=1).to(embed_weights.dtype)
#     one_hot_adv.requires_grad_() 
#     # Initialize the optimizer
#     optimizer = EGDwithAdamOptimizer([one_hot_adv], lr=step_size)
#     # Initialize the learning_rate scheduler
#     scheduler_cycle = 0
#     scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=50)
#     # Specify the filename
#     last_three_words = lambda text: '_'.join(text.split()[-3:])
#     first_three_words = lambda text: '_'.join(text.split()[0:3])
#     csv_filename = directories[0]+'/stats_('+first_three_words(user_prompt)+".."+last_three_words(user_prompt)+')('+str(num_steps)+'_steps).csv'        
#     # Generate column names
#     column_names = ['epoch', 'cycle', 'learning_rate', 'entropy_term', 
#                     'kl_divergence_term', 
#                     'continuous_loss','discrete_loss'] 
#     # Adding 'max_1' to 'max_20' column names using a loop
#     for i in range(1, 21):
#         column_names.append(f'max_{i}')
#     for i in range(1, 21):
#         column_names.append(f'token_id_{i}')
#     # Create an empty DataFrame
#     df = pd.DataFrame(columns=column_names)

#     # PHASE_2: bool=False
#     BREAK_IT: bool=False

#     for epoch_no in tqdm(range(num_steps)):
#         optimizer.zero_grad()

#         embeddings_adv = (one_hot_adv @ embed_weights).unsqueeze(0)
#         cross_entropy_loss, model_outputs = calc_loss(model, embeddings_user, embeddings_adv, embeddings_target, one_hot_target)
#         continuous_loss = cross_entropy_loss.detach().cpu().item()
#         # Break the loop if loss becomes "NaN"
#         if math.isnan(continuous_loss):
#             print("The value is NaN.")
#             BREAK_IT=True # Go to the Next Behavior

        
#         reg_coefficient = initial_coefficient * (final_coefficient / initial_coefficient) ** (epoch_no / (num_steps - 1))           
#         ################# Usual entropy #################
#         # # Calculate H(X) = − ∑_{i=1}^{L} ∑_{j=1}^{|T|} X_{ij} (log X_{ij} −1)
#         # Adding a small epsilon to avoid log(0) which is undefined
#         eps = 1e-12
#         one_hot_adv_float32 = one_hot_adv.to(dtype=torch.float32)
#         log_one_hot = torch.log(one_hot_adv_float32 + eps)
#         # Compute the modified entropy
#         entropy_per_row = -one_hot_adv_float32 * (log_one_hot - 1)
#         entropy_term = entropy_per_row.sum()
#         # Calculate the annealed coefficient, 𝜖
#         # Use Exponential Scheduling
#         entropy_term *= reg_coefficient 
#         # Take the Negative log of the highest probability in each row
#         peakiness_term = -torch.log(torch.max(one_hot_adv, dim=1).values + eps).sum()
#         peakiness_term *=reg_coefficient

#         # Experimental Starts
#         # ---- before backward(), after Cross-Entropy is computed ----
#         # # Ask the model for hidden states (final layer pre-unembedding)
#         # full = torch.hstack([embeddings_user, embeddings_adv]) # what comes after the adversarial suffix?
#         # outputs = model(inputs_embeds=full, output_hidden_states=True, use_cache=False)
#         # hidden  = model_outputs.hidden_states[-1]         # (1, L_total, H)
#         # h_last  = hidden[:, -1, :]                        # (1, H) last real position

#         # # Build a target embedding summary from the model's input embeddings
#         # e_tgt = embeddings_target.mean(dim=1)              # (1, E)


#         # Extract the last hidden states
#         hidden = model_outputs.hidden_states[-1]  # (batch, seq_len, hidden_dim)
#         L_u = embeddings_user.shape[1]
#         L_a = embeddings_adv.shape[1]
#         L_t = embeddings_target.shape[1]

#         # 1. index of last token of user+adv
#         idx_useradv_last = L_u + L_a - 1
#         # 2. index of first token of target
#         # idx_target_first = L_u + L_a

#         # Get corresponding hidden vectors
#         h_useradv_last = hidden[:, idx_useradv_last, :]   # (batch, hidden_dim)
#         # h_target_first = hidden[:, idx_target_first, :]   # (batch, hidden_dim)
#         # get embedding (NOT hidden) of the first target token
#         emb_target_first = embeddings_target[:, 0, :]           # (batch, D)

#         # === Compute MSE loss ===
#         lambda_reg = initial_lambda * (final_lambda / initial_lambda) ** (epoch_no / (num_steps - 1))
#         # mse_loss = nn.MSELoss()(h_useradv_last, h_target_first)
#         # Now compute MSE between final hidden vector of user+adv and the embedding of first target token
#         # mse_loss = nn.MSELoss()(h_useradv_last, emb_target_first)
#         # mse_loss *=lambda_reg
#         cosine_loss = 1.0 - torch.nn.functional.cosine_similarity(h_useradv_last, emb_target_first, dim=-1).mean()
#         cosine_loss*=lambda_reg
        
#         # Regularized loss: F(X) - H(X) + KL(X|~X) + reg_mse
#         regularized_loss = cross_entropy_loss - entropy_term + peakiness_term + cosine_loss 

#         # Experiment Ends
        
#         # Backpropagate the regularized loss
#         regularized_loss.backward()

#         # Get the scheduler's state to get learning_rate
#         scheduler_state = scheduler.state_dict()

#         # Clip the gradients(before update)
#         torch.nn.utils.clip_grad_norm_([one_hot_adv], max_norm=1.0)

#         # update parameters
#         optimizer.step()
#         # update scheduler
#         scheduler.step(continuous_loss)
        
#         # Discretization Starts
#         masked_one_hot_adv = get_masked_one_hot_adv(one_hot_adv)
#         # Use this masked_one_hot_adv ONLY for discretization
#         max_values = torch.max(masked_one_hot_adv, dim=1)
#         adv_token_ids = max_values.indices
#         # adv_token_ids = top_k_sampling(one_hot_adv, k=10).squeeze(dim=1)
#         # tau = initial_tau * (final_tau / initial_tau) ** (epoch_no / (num_steps - 1))
#         # one_hot_discrete = F.gumbel_softmax(one_hot_adv, tau=tau, hard=True)
#         # adv_token_ids = one_hot_discrete.argmax(dim=1)
#         one_hot_discrete = torch.zeros(
#             adv_token_ids.shape[0], embed_weights.shape[0], device=device, dtype=embed_weights.dtype
#         )
#         one_hot_discrete.scatter_(
#             1,
#             adv_token_ids.unsqueeze(1),
#             torch.ones(masked_one_hot_adv.shape[0], 1, device=device, dtype=embed_weights.dtype),
#         )
#         # Use one_hot_discrete to print Tokens
#         # What other techniques Can we use here to discretize the one_hot encodings?
#         # Use discrete tokens to calculate loss
#         with torch.no_grad():
#             embeddings_adv_discrete = (one_hot_discrete @ embed_weights).unsqueeze(0)
#             disc_loss, _ = calc_loss(model, embeddings_user, embeddings_adv_discrete, embeddings_target, one_hot_target)
#         # If loss improves, save it as x_best
#         discrete_loss =  disc_loss.detach().cpu().item()
#         # cur_loss_list.append(cur_loss)
#         if discrete_loss < best_disc_loss:
#             # print(f"########## {discrete_loss} #########")
#             best_disc_loss = discrete_loss
#             best_loss_at_epoch = epoch_no
#             effective_adv_embeddings = embeddings_adv_discrete
#             effective_adv_one_hot = one_hot_discrete
#         else :
#             pass
#         # # Discretization Ends
#         max_values = torch.max(masked_one_hot_adv, dim=1)
#         adv_token_ids = max_values.indices
#         # Dump Ouput values to a CSV file
#         # Convert max_values to a NumPy array
#         max_values_array = max_values.values.detach().cpu().numpy()
#         token_ids_array = adv_token_ids.detach().cpu().numpy()
#         # Get the scheduler's state and learning_rate
#         # Access the current learning rate directly from the optimizer's parameter group
#         scheduler_lr = optimizer.param_groups[0]['lr']  # Get the learning rate of the first parameter group
#         # scheduler_lr = scheduler_state['_last_lr'][0]
#         # Create the Initial array       
#         prepend_array = np.array([epoch_no, scheduler_cycle, scheduler_lr,
#                                   entropy_term.detach().cpu().item(), 
#                                   peakiness_term.detach().cpu().item(),
#                                   continuous_loss, discrete_loss]) 
        
#         # Concatenate the arrays
#         row = np.concatenate((prepend_array, max_values_array, token_ids_array))
#         new_row = pd.Series(row, index=df.columns)
#         df = pd.concat([df, new_row.to_frame().T], ignore_index=True)
#         # Save log data to CSV file periodically
#         if epoch_no % 10 == 0:
#             df.to_csv(csv_filename, index=False)
#         # Something went wrong; so break it and go to the next behavior.
#         if BREAK_IT==True:
#             break
#     # End of optimizations        
#     # Write to CSV file
#     df.to_csv(csv_filename, index=False)
#     # Token and Output Generation Step
#     inputs_token_ids = one_hot_inputs.argmax(dim=1)
#     adv_token_ids = effective_adv_one_hot.argmax(dim=1)
#     adv_suffix_list = str(adv_token_ids.cpu().numpy().tolist())
#     adv_suffix_string = tokenizer.decode(adv_token_ids.cpu().numpy())
#     final_string_ids = torch.hstack([inputs_token_ids, adv_token_ids])
#     outputs = []
#     generation_loop = range(1)
#     import warnings
#     # Suppress specific warnings
#     warnings.filterwarnings("ignore", message=".*`do_sample` is set to `False`. However, `temperature` is set to.*")
#     warnings.filterwarnings("ignore", message=".*`do_sample` is set to `False`. However, `top_p` is set to.*")
    
#     ######## Discrete tokens for Output Generation ########
#     for loop in tqdm(generation_loop, desc="Generating outputs", leave=False):
#         generated_output = model.generate(final_string_ids.unsqueeze(0), 
#                                           max_length=num_tokens, 
#                                           pad_token_id=tokenizer.pad_token_id,
#                                           do_sample=False)
#         generated_output_string = tokenizer.decode(generated_output[0][:].cpu().numpy(), skip_special_tokens=True).strip()
#         outputs.append(generated_output_string)
    
#     iteration_result = {
#         "harmful-behaviour": user_prompt,
#         "target": target,
#         "suffix_token_ids": adv_suffix_list,
#         "suffix_string": adv_suffix_string,
#         "epoch_no": epoch_no,
#         "continuous_loss": continuous_loss,
#         "discrete_loss": discrete_loss,
#         "best_disc_loss":best_disc_loss,
#         "epch_at_best_disc_loss": best_loss_at_epoch,
#         "outputs": outputs
#     }
#     optimization_results.append(iteration_result)

#     # Update the JSON File
#     with open(json_file_path, "w") as f:
#         json.dump(
#             optimization_results,
#             f,
#             indent=4,  # Keep structure readable
#             ensure_ascii=False  # Do this to Handle non-ASCII chars properly; If there is any
#         )

#     ##########################################################################################
#     ## Create and Update a JSONL file (required for running the parse_results.ipynb script) ##
#     output_file = directories[1]+"/output_"+model_name+"("+str(len(harmful_behaviors))+"_behaviors)("+str(num_steps)+"_steps).jsonl"
#     from behavior import Behavior
#     behavior = Behavior(user_prompt, adv_suffix_string, generated_output_string, "", "")
    
#     with open(output_file, 'a') as f:
#         f.write(json.dumps(behavior.to_dict()) + '\n')
#     f.close()  
#     ## Create and Update a JSONL file (required for running the parse_results.ipynb script) ##
#     ########################################################################################## s

In [ ]:
# # Update the one_hot encodings using Gradient Descent.
# from math import *
# import json 
# import pandas as pd
# import torch.nn.functional as F

# # Fetch the Model's weights
# embed_weights = get_embedding_matrix(model)
# # reader = [[user_prompt, target]]
# optimization_results = []
# # Set your regularization parameter
# initial_coefficient = 1e-5
# final_coefficient = 1e-3
# initial_lambda = 0.01
# final_lambda = 0.2

# # Create the following directories if they do NOT exist
# directories = [f'./CSV_Files/Single-Prompt/emb_target_mean(cosine)/{model_name}/{dataset_name}',       # 0
#                f'./JSON_Files/Single-Prompt/emb_target_mean(cosine)/{model_name}/{dataset_name}']      # 1

# for directory in directories:
#     # Create the directory if it doesn't exist
#     if not os.path.exists(directory):
#         os.makedirs(directory)
#         print(f"Directory '{directory}' created.")
#     else:
#         print(f"Directory '{directory}' already exists.")

    
# # JSON file to dump the model generated outputs 
# json_file_path = directories[1]+"/output_("+str(len(harmful_behaviors))+"_behaviors)("+str(num_steps)+"_steps).json"

# for row in tqdm(harmful_behaviors, desc="Optimizing prompts"):
#     iteration_result = {}
#     user_prompt, target = row
#     print(user_prompt)
#     # target = unsafe_target
#     user_prompt_tokens = get_tokens(user_prompt, tokenizer=tokenizer, device=device)    
#     target_tokens = get_tokens(target, tokenizer=tokenizer, device=device)[1:]
#     plotlosses = PlotLosses()
#     plotlosses = PlotLosses(groups={'loss': ['continuous_loss', 'discrete_loss']})    
#     target_tokens = get_tokens(target, tokenizer=tokenizer, device=device)[1:]

#     # Initialize one_hot encodings and embedding vectors for the user prompts and targets 
#     one_hot_inputs, embeddings_user = create_one_hot_and_embeddings(user_prompt_tokens, embed_weights, device=device)
#     one_hot_target, embeddings_target = create_one_hot_and_embeddings(target_tokens, embed_weights, device=device)
#     best_disc_loss = np.inf
#     best_loss_at_epoch = 0
#     # # Initialize the adversarial one_hot encodings using ONLY ASCII tokens
#     # one_hot_adv = get_only_ASCII_one_hot_adv()
#     # # Do NOT Initialize the adversarial one_hot encodings using all '!' tokens
#     # one_hot_adv = get_all_exclamations_one_hot_adv(embed_weights=embed_weights)
#     # You can also initialize one_hot_adv Randomly.
#     # one_hot_adv = F.softmax(torch.rand(20, tokenizer.vocab_size, dtype=torch.float16).to(device=device), dim=1).to(embed_weights.dtype)
#     one_hot_adv = F.softmax(torch.rand(20, embed_weights.shape[0], dtype=torch.float16).to(device=device), dim=1).to(embed_weights.dtype)
#     one_hot_adv.requires_grad_() 
#     # Initialize the optimizer
#     optimizer = EGDwithAdamOptimizer([one_hot_adv], lr=step_size)
#     # Initialize the learning_rate scheduler
#     scheduler_cycle = 0
#     scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=50)
#     # Specify the filename
#     last_three_words = lambda text: '_'.join(text.split()[-3:])
#     first_three_words = lambda text: '_'.join(text.split()[0:3])
#     csv_filename = directories[0]+'/stats_('+first_three_words(user_prompt)+".."+last_three_words(user_prompt)+')('+str(num_steps)+'_steps).csv'        
#     # Generate column names
#     column_names = ['epoch', 'cycle', 'learning_rate', 'entropy_term', 
#                     'kl_divergence_term', 
#                     'continuous_loss','discrete_loss'] 
#     # Adding 'max_1' to 'max_20' column names using a loop
#     for i in range(1, 21):
#         column_names.append(f'max_{i}')
#     for i in range(1, 21):
#         column_names.append(f'token_id_{i}')
#     # Create an empty DataFrame
#     df = pd.DataFrame(columns=column_names)

#     # PHASE_2: bool=False
#     BREAK_IT: bool=False

#     for epoch_no in tqdm(range(num_steps)):
#         optimizer.zero_grad()

#         embeddings_adv = (one_hot_adv @ embed_weights).unsqueeze(0)
#         cross_entropy_loss, model_outputs = calc_loss(model, embeddings_user, embeddings_adv, embeddings_target, one_hot_target)
#         continuous_loss = cross_entropy_loss.detach().cpu().item()
#         # Break the loop if loss becomes "NaN"
#         if math.isnan(continuous_loss):
#             print("The value is NaN.")
#             BREAK_IT=True # Go to the Next Behavior

        
#         reg_coefficient = initial_coefficient * (final_coefficient / initial_coefficient) ** (epoch_no / (num_steps - 1))           
#         ################# Usual entropy #################
#         # # Calculate H(X) = − ∑_{i=1}^{L} ∑_{j=1}^{|T|} X_{ij} (log X_{ij} −1)
#         # Adding a small epsilon to avoid log(0) which is undefined
#         eps = 1e-12
#         one_hot_adv_float32 = one_hot_adv.to(dtype=torch.float32)
#         log_one_hot = torch.log(one_hot_adv_float32 + eps)
#         # Compute the modified entropy
#         entropy_per_row = -one_hot_adv_float32 * (log_one_hot - 1)
#         entropy_term = entropy_per_row.sum()
#         # Calculate the annealed coefficient, 𝜖
#         # Use Exponential Scheduling
#         entropy_term *= reg_coefficient 
#         # Take the Negative log of the highest probability in each row
#         peakiness_term = -torch.log(torch.max(one_hot_adv, dim=1).values + eps).sum()
#         peakiness_term *=reg_coefficient

#         # Experimental Starts
#         # ---- before backward(), after Cross-Entropy is computed ----
#         # # Ask the model for hidden states (final layer pre-unembedding)
#         # full = torch.hstack([embeddings_user, embeddings_adv]) # what comes after the adversarial suffix?
#         # outputs = model(inputs_embeds=full, output_hidden_states=True, use_cache=False)
#         # hidden  = model_outputs.hidden_states[-1]         # (1, L_total, H)
#         # h_last  = hidden[:, -1, :]                        # (1, H) last real position

#         # Build a target embedding summary from the model's input embeddings
#         emb_target_mean = embeddings_target.mean(dim=1)              # (1, E)

#         # # Choose your penalty:
#         # lambda_reg = initial_lambda * (final_lambda / initial_lambda) ** (epoch_no / (num_steps - 1))
#         # # reg_mse = torch.nn.functional.mse_loss(h_last, e_tgt)
#         # # or:
        
#         # Extract the last hidden states
#         hidden = model_outputs.hidden_states[-1]  # (batch, seq_len, hidden_dim)
#         L_u = embeddings_user.shape[1]
#         L_a = embeddings_adv.shape[1]

#         # 1. index of last token of user+adv
#         idx_useradv_last = L_u + L_a - 1
#         # 2. index of first token of target
#         # idx_target_first = L_u + L_a

#         # Get corresponding hidden vectors
#         h_useradv_last = hidden[:, idx_useradv_last, :]   # (batch, hidden_dim)
#         # h_target_first = hidden[:, idx_target_first, :]   # (batch, hidden_dim)
#         # get embedding (NOT hidden) of the first target token
#         # emb_target_first = embeddings_target[:, 0, :]           # (batch, D)

#         # === Compute MSE loss ===
#         lambda_reg = initial_lambda * (final_lambda / initial_lambda) ** (epoch_no / (num_steps - 1))
#         # mse_loss = nn.MSELoss()(h_useradv_last, h_target_first)
#         # Now compute MSE between final hidden vector of user+adv and the embedding of first target token
#         # mse_loss = nn.MSELoss()(h_useradv_last, emb_target_mean)
#         # mse_loss *=lambda_reg
#         cosine_loss = 1.0 - torch.nn.functional.cosine_similarity(h_useradv_last, emb_target_mean, dim=-1).mean()
#         cosine_loss*=lambda_reg
        
#         # Regularized loss: F(X) - H(X) + KL(X|~X) + reg_mse
#         regularized_loss = cross_entropy_loss - entropy_term + peakiness_term +cosine_loss 

#         # Experiment Ends
        
#         # Backpropagate the regularized loss
#         regularized_loss.backward()

#         # Get the scheduler's state to get learning_rate
#         scheduler_state = scheduler.state_dict()

#         # Clip the gradients(before update)
#         torch.nn.utils.clip_grad_norm_([one_hot_adv], max_norm=1.0)

#         # update parameters
#         optimizer.step()
#         # update scheduler
#         scheduler.step(continuous_loss)
        
#         # Discretization Starts
#         masked_one_hot_adv = get_masked_one_hot_adv(one_hot_adv)
#         # Use this masked_one_hot_adv ONLY for discretization
#         max_values = torch.max(masked_one_hot_adv, dim=1)
#         adv_token_ids = max_values.indices
#         # adv_token_ids = top_k_sampling(one_hot_adv, k=10).squeeze(dim=1)
#         # tau = initial_tau * (final_tau / initial_tau) ** (epoch_no / (num_steps - 1))
#         # one_hot_discrete = F.gumbel_softmax(one_hot_adv, tau=tau, hard=True)
#         # adv_token_ids = one_hot_discrete.argmax(dim=1)
#         one_hot_discrete = torch.zeros(
#             adv_token_ids.shape[0], embed_weights.shape[0], device=device, dtype=embed_weights.dtype
#         )
#         one_hot_discrete.scatter_(
#             1,
#             adv_token_ids.unsqueeze(1),
#             torch.ones(masked_one_hot_adv.shape[0], 1, device=device, dtype=embed_weights.dtype),
#         )
#         # Use one_hot_discrete to print Tokens
#         # What other techniques Can we use here to discretize the one_hot encodings?
#         # Use discrete tokens to calculate loss
#         with torch.no_grad():
#             embeddings_adv_discrete = (one_hot_discrete @ embed_weights).unsqueeze(0)
#             disc_loss, _ = calc_loss(model, embeddings_user, embeddings_adv_discrete, embeddings_target, one_hot_target)
#         # If loss improves, save it as x_best
#         discrete_loss =  disc_loss.detach().cpu().item()
#         # cur_loss_list.append(cur_loss)
#         if discrete_loss < best_disc_loss:
#             # print(f"########## {discrete_loss} #########")
#             best_disc_loss = discrete_loss
#             best_loss_at_epoch = epoch_no
#             effective_adv_embeddings = embeddings_adv_discrete
#             effective_adv_one_hot = one_hot_discrete
#         else :
#             pass
#         # # Discretization Ends
#         max_values = torch.max(masked_one_hot_adv, dim=1)
#         adv_token_ids = max_values.indices
#         # Dump Ouput values to a CSV file
#         # Convert max_values to a NumPy array
#         max_values_array = max_values.values.detach().cpu().numpy()
#         token_ids_array = adv_token_ids.detach().cpu().numpy()
#         # Get the scheduler's state and learning_rate
#         # Access the current learning rate directly from the optimizer's parameter group
#         scheduler_lr = optimizer.param_groups[0]['lr']  # Get the learning rate of the first parameter group
#         # scheduler_lr = scheduler_state['_last_lr'][0]
#         # Create the Initial array       
#         prepend_array = np.array([epoch_no, scheduler_cycle, scheduler_lr,
#                                   entropy_term.detach().cpu().item(), 
#                                   peakiness_term.detach().cpu().item(),
#                                   continuous_loss, discrete_loss]) 
        
#         # Concatenate the arrays
#         row = np.concatenate((prepend_array, max_values_array, token_ids_array))
#         new_row = pd.Series(row, index=df.columns)
#         df = pd.concat([df, new_row.to_frame().T], ignore_index=True)
#         # Save log data to CSV file periodically
#         if epoch_no % 10 == 0:
#             df.to_csv(csv_filename, index=False)
#         # Something went wrong; so break it and go to the next behavior.
#         if BREAK_IT==True:
#             break
#     # End of optimizations        
#     # Write to CSV file
#     df.to_csv(csv_filename, index=False)
#     # Token and Output Generation Step
#     inputs_token_ids = one_hot_inputs.argmax(dim=1)
#     adv_token_ids = effective_adv_one_hot.argmax(dim=1)
#     adv_suffix_list = str(adv_token_ids.cpu().numpy().tolist())
#     adv_suffix_string = tokenizer.decode(adv_token_ids.cpu().numpy())
#     final_string_ids = torch.hstack([inputs_token_ids, adv_token_ids])
#     outputs = []
#     generation_loop = range(1)
#     import warnings
#     # Suppress specific warnings
#     warnings.filterwarnings("ignore", message=".*`do_sample` is set to `False`. However, `temperature` is set to.*")
#     warnings.filterwarnings("ignore", message=".*`do_sample` is set to `False`. However, `top_p` is set to.*")
    
#     ######## Discrete tokens for Output Generation ########
#     for loop in tqdm(generation_loop, desc="Generating outputs", leave=False):
#         generated_output = model.generate(final_string_ids.unsqueeze(0), 
#                                           max_length=num_tokens, 
#                                           pad_token_id=tokenizer.pad_token_id,
#                                           do_sample=False)
#         generated_output_string = tokenizer.decode(generated_output[0][:].cpu().numpy(), skip_special_tokens=True).strip()
#         outputs.append(generated_output_string)
    
#     iteration_result = {
#         "harmful-behaviour": user_prompt,
#         "target": target,
#         "suffix_token_ids": adv_suffix_list,
#         "suffix_string": adv_suffix_string,
#         "epoch_no": epoch_no,
#         "continuous_loss": continuous_loss,
#         "discrete_loss": discrete_loss,
#         "best_disc_loss":best_disc_loss,
#         "epch_at_best_disc_loss": best_loss_at_epoch,
#         "outputs": outputs
#     }
#     optimization_results.append(iteration_result)

#     # Update the JSON File
#     with open(json_file_path, "w") as f:
#         json.dump(
#             optimization_results,
#             f,
#             indent=4,  # Keep structure readable
#             ensure_ascii=False  # Do this to Handle non-ASCII chars properly; If there is any
#         )

#     ##########################################################################################
#     ## Create and Update a JSONL file (required for running the parse_results.ipynb script) ##
#     output_file = directories[1]+"/output_"+model_name+"("+str(len(harmful_behaviors))+"_behaviors)("+str(num_steps)+"_steps).jsonl"
#     from behavior import Behavior
#     behavior = Behavior(user_prompt, adv_suffix_string, generated_output_string, "", "")
    
#     with open(output_file, 'a') as f:
#         f.write(json.dumps(behavior.to_dict()) + '\n')
#     f.close()  
#     ## Create and Update a JSONL file (required for running the parse_results.ipynb script) ##
#     ########################################################################################## s

In [ ]:
# # Update the one_hot encodings using Gradient Descent.
# from math import *
# import json 
# import pandas as pd
# import torch.nn.functional as F

# # Fetch the Model's weights
# embed_weights = get_embedding_matrix(model)
# # reader = [[user_prompt, target]]
# optimization_results = []
# # Set your regularization parameter
# initial_coefficient = 1e-5
# final_coefficient = 1e-3
# initial_lambda = 0.01
# final_lambda = 0.2

# # Create the following directories if they do NOT exist
# directories = [f'./CSV_Files/Single-Prompt/emb_target_mean_forward(cosine)/{model_name}/{dataset_name}',       # 0
#                f'./JSON_Files/Single-Prompt/emb_target_mean_forward(cosine)/{model_name}/{dataset_name}']      # 1

# for directory in directories:
#     # Create the directory if it doesn't exist
#     if not os.path.exists(directory):
#         os.makedirs(directory)
#         print(f"Directory '{directory}' created.")
#     else:
#         print(f"Directory '{directory}' already exists.")

    
# # JSON file to dump the model generated outputs 
# json_file_path = directories[1]+"/output_("+str(len(harmful_behaviors))+"_behaviors)("+str(num_steps)+"_steps).json"

# for row in tqdm(harmful_behaviors, desc="Optimizing prompts"):
#     iteration_result = {}
#     user_prompt, target = row
#     print(user_prompt)
#     # target = unsafe_target
#     user_prompt_tokens = get_tokens(user_prompt, tokenizer=tokenizer, device=device)    
#     target_tokens = get_tokens(target, tokenizer=tokenizer, device=device)[1:]
#     plotlosses = PlotLosses()
#     plotlosses = PlotLosses(groups={'loss': ['continuous_loss', 'discrete_loss']})    
#     target_tokens = get_tokens(target, tokenizer=tokenizer, device=device)[1:]

#     # Initialize one_hot encodings and embedding vectors for the user prompts and targets 
#     one_hot_inputs, embeddings_user = create_one_hot_and_embeddings(user_prompt_tokens, embed_weights, device=device)
#     one_hot_target, embeddings_target = create_one_hot_and_embeddings(target_tokens, embed_weights, device=device)
#     best_disc_loss = np.inf
#     best_loss_at_epoch = 0
#     # # Initialize the adversarial one_hot encodings using ONLY ASCII tokens
#     # one_hot_adv = get_only_ASCII_one_hot_adv()
#     # # Do NOT Initialize the adversarial one_hot encodings using all '!' tokens
#     # one_hot_adv = get_all_exclamations_one_hot_adv(embed_weights=embed_weights)
#     # You can also initialize one_hot_adv Randomly.
#     # one_hot_adv = F.softmax(torch.rand(20, tokenizer.vocab_size, dtype=torch.float16).to(device=device), dim=1).to(embed_weights.dtype)
#     one_hot_adv = F.softmax(torch.rand(20, embed_weights.shape[0], dtype=torch.float16).to(device=device), dim=1).to(embed_weights.dtype)
#     one_hot_adv.requires_grad_() 
#     # Initialize the optimizer
#     optimizer = EGDwithAdamOptimizer([one_hot_adv], lr=step_size)
#     # Initialize the learning_rate scheduler
#     scheduler_cycle = 0
#     scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=50)
#     # Specify the filename
#     last_three_words = lambda text: '_'.join(text.split()[-3:])
#     first_three_words = lambda text: '_'.join(text.split()[0:3])
#     csv_filename = directories[0]+'/stats_('+first_three_words(user_prompt)+".."+last_three_words(user_prompt)+')('+str(num_steps)+'_steps).csv'        
#     # Generate column names
#     column_names = ['epoch', 'cycle', 'learning_rate', 'entropy_term', 
#                     'kl_divergence_term', 
#                     'continuous_loss','discrete_loss'] 
#     # Adding 'max_1' to 'max_20' column names using a loop
#     for i in range(1, 21):
#         column_names.append(f'max_{i}')
#     for i in range(1, 21):
#         column_names.append(f'token_id_{i}')
#     # Create an empty DataFrame
#     df = pd.DataFrame(columns=column_names)

#     # PHASE_2: bool=False
#     BREAK_IT: bool=False

#     for epoch_no in tqdm(range(num_steps)):
#         optimizer.zero_grad()

#         embeddings_adv = (one_hot_adv @ embed_weights).unsqueeze(0)
#         cross_entropy_loss, _ = calc_loss(model, embeddings_user, embeddings_adv, embeddings_target, one_hot_target)
#         continuous_loss = cross_entropy_loss.detach().cpu().item()
#         # Break the loop if loss becomes "NaN"
#         if math.isnan(continuous_loss):
#             print("The value is NaN.")
#             BREAK_IT=True # Go to the Next Behavior

        
#         reg_coefficient = initial_coefficient * (final_coefficient / initial_coefficient) ** (epoch_no / (num_steps - 1))           
#         ################# Usual entropy #################
#         # # Calculate H(X) = − ∑_{i=1}^{L} ∑_{j=1}^{|T|} X_{ij} (log X_{ij} −1)
#         # Adding a small epsilon to avoid log(0) which is undefined
#         eps = 1e-12
#         one_hot_adv_float32 = one_hot_adv.to(dtype=torch.float32)
#         log_one_hot = torch.log(one_hot_adv_float32 + eps)
#         # Compute the modified entropy
#         entropy_per_row = -one_hot_adv_float32 * (log_one_hot - 1)
#         entropy_term = entropy_per_row.sum()
#         # Calculate the annealed coefficient, 𝜖
#         # Use Exponential Scheduling
#         entropy_term *= reg_coefficient 
#         # Take the Negative log of the highest probability in each row
#         peakiness_term = -torch.log(torch.max(one_hot_adv, dim=1).values + eps).sum()
#         peakiness_term *=reg_coefficient

#         # Experimental Starts
#         # ---- before backward(), after Cross-Entropy is computed ----
#         # Ask the model for hidden states (final layer pre-unembedding)
#         with torch.no_grad():
#             full_embeds = torch.hstack([embeddings_user, embeddings_adv]) # what comes after the adversarial suffix?
#             model_outputs = model(inputs_embeds=full_embeds, output_hidden_states=True, use_cache=False)
#         hidden  = model_outputs.hidden_states[-1]         # (1, L_total, H)
#         emb_hidden_last = hidden[:, -1, :]                        # (1, H) last real position

#         # Build a target embedding summary from the model's input embeddings
#         emb_target_mean = embeddings_target.mean(dim=1)              # (1, E)

#         # # Choose your penalty:
#         # lambda_reg = initial_lambda * (final_lambda / initial_lambda) ** (epoch_no / (num_steps - 1))
#         # # reg_mse = torch.nn.functional.mse_loss(h_last, e_tgt)
#         # # or:
#         # Extract the last hidden states
       

#         # Get corresponding hidden vectors
#         # h_useradv_last = hidden[:, idx_useradv_last, :]   # (batch, hidden_dim)
#         # h_target_first = hidden[:, idx_target_first, :]   # (batch, hidden_dim)
#         # get embedding (NOT hidden) of the first target token
#         # emb_target_first = embeddings_target[:, 0, :]           # (batch, D)

#         # === Compute MSE loss ===
#         lambda_reg = initial_lambda * (final_lambda / initial_lambda) ** (epoch_no / (num_steps - 1))
#         # mse_loss = nn.MSELoss()(h_useradv_last, h_target_first)
#         # Now compute MSE between final hidden vector of user+adv and the embedding of first target token
#         # mse_loss = nn.MSELoss()(emb_hidden_last, emb_target_mean)
#         # mse_loss *=lambda_reg
#         cosine_loss = 1.0 - torch.nn.functional.cosine_similarity(emb_hidden_last, emb_target_mean, dim=-1).mean()
#         cosine_loss*=lambda_reg

        
#         # Regularized loss: F(X) - H(X) + KL(X|~X) + reg_mse
#         regularized_loss = cross_entropy_loss - entropy_term + peakiness_term +cosine_loss 

#         # Experiment Ends
        
#         # Backpropagate the regularized loss
#         regularized_loss.backward()

#         # Get the scheduler's state to get learning_rate
#         scheduler_state = scheduler.state_dict()

#         # Clip the gradients(before update)
#         torch.nn.utils.clip_grad_norm_([one_hot_adv], max_norm=1.0)

#         # update parameters
#         optimizer.step()
#         # update scheduler
#         scheduler.step(continuous_loss)
        
#         # Discretization Starts
#         masked_one_hot_adv = get_masked_one_hot_adv(one_hot_adv)
#         # Use this masked_one_hot_adv ONLY for discretization
#         max_values = torch.max(masked_one_hot_adv, dim=1)
#         adv_token_ids = max_values.indices
#         # adv_token_ids = top_k_sampling(one_hot_adv, k=10).squeeze(dim=1)
#         # tau = initial_tau * (final_tau / initial_tau) ** (epoch_no / (num_steps - 1))
#         # one_hot_discrete = F.gumbel_softmax(one_hot_adv, tau=tau, hard=True)
#         # adv_token_ids = one_hot_discrete.argmax(dim=1)
#         one_hot_discrete = torch.zeros(
#             adv_token_ids.shape[0], embed_weights.shape[0], device=device, dtype=embed_weights.dtype
#         )
#         one_hot_discrete.scatter_(
#             1,
#             adv_token_ids.unsqueeze(1),
#             torch.ones(masked_one_hot_adv.shape[0], 1, device=device, dtype=embed_weights.dtype),
#         )
#         # Use one_hot_discrete to print Tokens
#         # What other techniques Can we use here to discretize the one_hot encodings?
#         # Use discrete tokens to calculate loss
#         with torch.no_grad():
#             embeddings_adv_discrete = (one_hot_discrete @ embed_weights).unsqueeze(0)
#             disc_loss, _ = calc_loss(model, embeddings_user, embeddings_adv_discrete, embeddings_target, one_hot_target)
#         # If loss improves, save it as x_best
#         discrete_loss =  disc_loss.detach().cpu().item()
#         # cur_loss_list.append(cur_loss)
#         if discrete_loss < best_disc_loss:
#             # print(f"########## {discrete_loss} #########")
#             best_disc_loss = discrete_loss
#             best_loss_at_epoch = epoch_no
#             effective_adv_embeddings = embeddings_adv_discrete
#             effective_adv_one_hot = one_hot_discrete
#         else :
#             pass
#         # # Discretization Ends
#         max_values = torch.max(masked_one_hot_adv, dim=1)
#         adv_token_ids = max_values.indices
#         # Dump Ouput values to a CSV file
#         # Convert max_values to a NumPy array
#         max_values_array = max_values.values.detach().cpu().numpy()
#         token_ids_array = adv_token_ids.detach().cpu().numpy()
#         # Get the scheduler's state and learning_rate
#         # Access the current learning rate directly from the optimizer's parameter group
#         scheduler_lr = optimizer.param_groups[0]['lr']  # Get the learning rate of the first parameter group
#         # scheduler_lr = scheduler_state['_last_lr'][0]
#         # Create the Initial array       
#         prepend_array = np.array([epoch_no, scheduler_cycle, scheduler_lr,
#                                   entropy_term.detach().cpu().item(), 
#                                   peakiness_term.detach().cpu().item(),
#                                   continuous_loss, discrete_loss]) 
        
#         # Concatenate the arrays
#         row = np.concatenate((prepend_array, max_values_array, token_ids_array))
#         new_row = pd.Series(row, index=df.columns)
#         df = pd.concat([df, new_row.to_frame().T], ignore_index=True)
#         # Save log data to CSV file periodically
#         if epoch_no % 10 == 0:
#             df.to_csv(csv_filename, index=False)
#         # Something went wrong; so break it and go to the next behavior.
#         if BREAK_IT==True:
#             break
#     # End of optimizations        
#     # Write to CSV file
#     df.to_csv(csv_filename, index=False)
#     # Token and Output Generation Step
#     inputs_token_ids = one_hot_inputs.argmax(dim=1)
#     adv_token_ids = effective_adv_one_hot.argmax(dim=1)
#     adv_suffix_list = str(adv_token_ids.cpu().numpy().tolist())
#     adv_suffix_string = tokenizer.decode(adv_token_ids.cpu().numpy())
#     final_string_ids = torch.hstack([inputs_token_ids, adv_token_ids])
#     outputs = []
#     generation_loop = range(1)
#     import warnings
#     # Suppress specific warnings
#     warnings.filterwarnings("ignore", message=".*`do_sample` is set to `False`. However, `temperature` is set to.*")
#     warnings.filterwarnings("ignore", message=".*`do_sample` is set to `False`. However, `top_p` is set to.*")
    
#     ######## Discrete tokens for Output Generation ########
#     for loop in tqdm(generation_loop, desc="Generating outputs", leave=False):
#         generated_output = model.generate(final_string_ids.unsqueeze(0), 
#                                           max_length=num_tokens, 
#                                           pad_token_id=tokenizer.pad_token_id,
#                                           do_sample=False)
#         generated_output_string = tokenizer.decode(generated_output[0][:].cpu().numpy(), skip_special_tokens=True).strip()
#         outputs.append(generated_output_string)
    
#     iteration_result = {
#         "harmful-behaviour": user_prompt,
#         "target": target,
#         "suffix_token_ids": adv_suffix_list,
#         "suffix_string": adv_suffix_string,
#         "epoch_no": epoch_no,
#         "continuous_loss": continuous_loss,
#         "discrete_loss": discrete_loss,
#         "best_disc_loss":best_disc_loss,
#         "epch_at_best_disc_loss": best_loss_at_epoch,
#         "outputs": outputs
#     }
#     optimization_results.append(iteration_result)

#     # Update the JSON File
#     with open(json_file_path, "w") as f:
#         json.dump(
#             optimization_results,
#             f,
#             indent=4,  # Keep structure readable
#             ensure_ascii=False  # Do this to Handle non-ASCII chars properly; If there is any
#         )

#     ##########################################################################################
#     ## Create and Update a JSONL file (required for running the parse_results.ipynb script) ##
#     output_file = directories[1]+"/output_"+model_name+"("+str(len(harmful_behaviors))+"_behaviors)("+str(num_steps)+"_steps).jsonl"
#     from behavior import Behavior
#     behavior = Behavior(user_prompt, adv_suffix_string, generated_output_string, "", "")
    
#     with open(output_file, 'a') as f:
#         f.write(json.dumps(behavior.to_dict()) + '\n')
#     f.close()  
#     ## Create and Update a JSONL file (required for running the parse_results.ipynb script) ##
#     ########################################################################################## s

In [ ]:
# # Update the one_hot encodings using Gradient Descent.
# from math import *
# import json 
# import pandas as pd
# import torch.nn.functional as F

# # Fetch the Model's weights
# embed_weights = get_embedding_matrix(model)
# # reader = [[user_prompt, target]]
# optimization_results = []
# # Set your regularization parameter
# initial_coefficient = 1e-5
# final_coefficient = 1e-3
# initial_lambda = 0.01
# final_lambda = 0.2

# # Create the following directories if they do NOT exist
# directories = [f'./CSV_Files/Single-Prompt/emb_target_first_forward(cosine)/{model_name}/{dataset_name}',       # 0
#                f'./JSON_Files/Single-Prompt/emb_target_first_forward(cosine)/{model_name}/{dataset_name}']      # 1

# for directory in directories:
#     # Create the directory if it doesn't exist
#     if not os.path.exists(directory):
#         os.makedirs(directory)
#         print(f"Directory '{directory}' created.")
#     else:
#         print(f"Directory '{directory}' already exists.")

    
# # JSON file to dump the model generated outputs 
# json_file_path = directories[1]+"/output_("+str(len(harmful_behaviors))+"_behaviors)("+str(num_steps)+"_steps).json"

# for row in tqdm(harmful_behaviors, desc="Optimizing prompts"):
#     iteration_result = {}
#     user_prompt, target = row
#     print(user_prompt)
#     # target = unsafe_target
#     user_prompt_tokens = get_tokens(user_prompt, tokenizer=tokenizer, device=device)    
#     target_tokens = get_tokens(target, tokenizer=tokenizer, device=device)[1:]
#     plotlosses = PlotLosses()
#     plotlosses = PlotLosses(groups={'loss': ['continuous_loss', 'discrete_loss']})    
#     target_tokens = get_tokens(target, tokenizer=tokenizer, device=device)[1:]

#     # Initialize one_hot encodings and embedding vectors for the user prompts and targets 
#     one_hot_inputs, embeddings_user = create_one_hot_and_embeddings(user_prompt_tokens, embed_weights, device=device)
#     one_hot_target, embeddings_target = create_one_hot_and_embeddings(target_tokens, embed_weights, device=device)
#     best_disc_loss = np.inf
#     best_loss_at_epoch = 0
#     # # Initialize the adversarial one_hot encodings using ONLY ASCII tokens
#     # one_hot_adv = get_only_ASCII_one_hot_adv()
#     # # Do NOT Initialize the adversarial one_hot encodings using all '!' tokens
#     # one_hot_adv = get_all_exclamations_one_hot_adv(embed_weights=embed_weights)
#     # You can also initialize one_hot_adv Randomly.
#     # one_hot_adv = F.softmax(torch.rand(20, tokenizer.vocab_size, dtype=torch.float16).to(device=device), dim=1).to(embed_weights.dtype)
#     one_hot_adv = F.softmax(torch.rand(20, embed_weights.shape[0], dtype=torch.float16).to(device=device), dim=1).to(embed_weights.dtype)
#     one_hot_adv.requires_grad_() 
#     # Initialize the optimizer
#     optimizer = EGDwithAdamOptimizer([one_hot_adv], lr=step_size)
#     # Initialize the learning_rate scheduler
#     scheduler_cycle = 0
#     scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=50)
#     # Specify the filename
#     last_three_words = lambda text: '_'.join(text.split()[-3:])
#     first_three_words = lambda text: '_'.join(text.split()[0:3])
#     csv_filename = directories[0]+'/stats_('+first_three_words(user_prompt)+".."+last_three_words(user_prompt)+')('+str(num_steps)+'_steps).csv'        
#     # Generate column names
#     column_names = ['epoch', 'cycle', 'learning_rate', 'entropy_term', 
#                     'kl_divergence_term', 
#                     'continuous_loss','discrete_loss'] 
#     # Adding 'max_1' to 'max_20' column names using a loop
#     for i in range(1, 21):
#         column_names.append(f'max_{i}')
#     for i in range(1, 21):
#         column_names.append(f'token_id_{i}')
#     # Create an empty DataFrame
#     df = pd.DataFrame(columns=column_names)

#     # PHASE_2: bool=False
#     BREAK_IT: bool=False

#     for epoch_no in tqdm(range(num_steps)):
#         optimizer.zero_grad()

#         embeddings_adv = (one_hot_adv @ embed_weights).unsqueeze(0)
#         cross_entropy_loss, _ = calc_loss(model, embeddings_user, embeddings_adv, embeddings_target, one_hot_target)
#         continuous_loss = cross_entropy_loss.detach().cpu().item()
#         # Break the loop if loss becomes "NaN"
#         if math.isnan(continuous_loss):
#             print("The value is NaN.")
#             BREAK_IT=True # Go to the Next Behavior

        
#         reg_coefficient = initial_coefficient * (final_coefficient / initial_coefficient) ** (epoch_no / (num_steps - 1))           
#         ################# Usual entropy #################
#         # # Calculate H(X) = − ∑_{i=1}^{L} ∑_{j=1}^{|T|} X_{ij} (log X_{ij} −1)
#         # Adding a small epsilon to avoid log(0) which is undefined
#         eps = 1e-12
#         one_hot_adv_float32 = one_hot_adv.to(dtype=torch.float32)
#         log_one_hot = torch.log(one_hot_adv_float32 + eps)
#         # Compute the modified entropy
#         entropy_per_row = -one_hot_adv_float32 * (log_one_hot - 1)
#         entropy_term = entropy_per_row.sum()
#         # Calculate the annealed coefficient, 𝜖
#         # Use Exponential Scheduling
#         entropy_term *= reg_coefficient 
#         # Take the Negative log of the highest probability in each row
#         peakiness_term = -torch.log(torch.max(one_hot_adv, dim=1).values + eps).sum()
#         peakiness_term *=reg_coefficient

#         # Experimental Starts
#         # ---- before backward(), after Cross-Entropy is computed ----
#         # Ask the model for hidden states (final layer pre-unembedding)
#         with torch.no_grad():
#             full_embeds = torch.hstack([embeddings_user, embeddings_adv]) # what comes after the adversarial suffix?
#             model_outputs = model(inputs_embeds=full_embeds, output_hidden_states=True, use_cache=False)
#         hidden  = model_outputs.hidden_states[-1]         # (1, L_total, H)
#         emb_hidden_last = hidden[:, -1, :]                        # (1, H) last real position

#         # Build a target embedding summary from the model's input embeddings
#         # emb_target_mean = embeddings_target.mean(dim=1)              # (1, E)

#         # # Choose your penalty:
#         # lambda_reg = initial_lambda * (final_lambda / initial_lambda) ** (epoch_no / (num_steps - 1))
#         # # reg_mse = torch.nn.functional.mse_loss(h_last, e_tgt)
#         # # or:
#         # Extract the last hidden states
       

#         # Get corresponding hidden vectors
#         # h_useradv_last = hidden[:, idx_useradv_last, :]   # (batch, hidden_dim)
#         # h_target_first = hidden[:, idx_target_first, :]   # (batch, hidden_dim)
#         # get embedding (NOT hidden) of the first target token
#         emb_target_first = embeddings_target[:, 0, :]           # (batch, D)

#         # === Compute MSE loss ===
#         lambda_reg = initial_lambda * (final_lambda / initial_lambda) ** (epoch_no / (num_steps - 1))
#         # mse_loss = nn.MSELoss()(h_useradv_last, h_target_first)
#         # Now compute MSE between final hidden vector of user+adv and the embedding of first target token
#         # mse_loss = nn.MSELoss()(emb_hidden_last, emb_target_first)
#         # mse_loss *=lambda_reg
#         cosine_loss = 1.0 - torch.nn.functional.cosine_similarity(emb_hidden_last, emb_target_first, dim=-1).mean()
#         cosine_loss*=lambda_reg
        
#         # Regularized loss: F(X) - H(X) + KL(X|~X) + reg_mse
#         regularized_loss = cross_entropy_loss - entropy_term + peakiness_term + cosine_loss 

#         # Experiment Ends
        
#         # Backpropagate the regularized loss
#         regularized_loss.backward()

#         # Get the scheduler's state to get learning_rate
#         scheduler_state = scheduler.state_dict()

#         # Clip the gradients(before update)
#         torch.nn.utils.clip_grad_norm_([one_hot_adv], max_norm=1.0)

#         # update parameters
#         optimizer.step()
#         # update scheduler
#         scheduler.step(continuous_loss)
        
#         # Discretization Starts
#         masked_one_hot_adv = get_masked_one_hot_adv(one_hot_adv)
#         # Use this masked_one_hot_adv ONLY for discretization
#         max_values = torch.max(masked_one_hot_adv, dim=1)
#         adv_token_ids = max_values.indices
#         # adv_token_ids = top_k_sampling(one_hot_adv, k=10).squeeze(dim=1)
#         # tau = initial_tau * (final_tau / initial_tau) ** (epoch_no / (num_steps - 1))
#         # one_hot_discrete = F.gumbel_softmax(one_hot_adv, tau=tau, hard=True)
#         # adv_token_ids = one_hot_discrete.argmax(dim=1)
#         one_hot_discrete = torch.zeros(
#             adv_token_ids.shape[0], embed_weights.shape[0], device=device, dtype=embed_weights.dtype
#         )
#         one_hot_discrete.scatter_(
#             1,
#             adv_token_ids.unsqueeze(1),
#             torch.ones(masked_one_hot_adv.shape[0], 1, device=device, dtype=embed_weights.dtype),
#         )
#         # Use one_hot_discrete to print Tokens
#         # What other techniques Can we use here to discretize the one_hot encodings?
#         # Use discrete tokens to calculate loss
#         with torch.no_grad():
#             embeddings_adv_discrete = (one_hot_discrete @ embed_weights).unsqueeze(0)
#             disc_loss, _ = calc_loss(model, embeddings_user, embeddings_adv_discrete, embeddings_target, one_hot_target)
#         # If loss improves, save it as x_best
#         discrete_loss =  disc_loss.detach().cpu().item()
#         # cur_loss_list.append(cur_loss)
#         if discrete_loss < best_disc_loss:
#             # print(f"########## {discrete_loss} #########")
#             best_disc_loss = discrete_loss
#             best_loss_at_epoch = epoch_no
#             effective_adv_embeddings = embeddings_adv_discrete
#             effective_adv_one_hot = one_hot_discrete
#         else :
#             pass
#         # # Discretization Ends
#         max_values = torch.max(masked_one_hot_adv, dim=1)
#         adv_token_ids = max_values.indices
#         # Dump Ouput values to a CSV file
#         # Convert max_values to a NumPy array
#         max_values_array = max_values.values.detach().cpu().numpy()
#         token_ids_array = adv_token_ids.detach().cpu().numpy()
#         # Get the scheduler's state and learning_rate
#         # Access the current learning rate directly from the optimizer's parameter group
#         scheduler_lr = optimizer.param_groups[0]['lr']  # Get the learning rate of the first parameter group
#         # scheduler_lr = scheduler_state['_last_lr'][0]
#         # Create the Initial array       
#         prepend_array = np.array([epoch_no, scheduler_cycle, scheduler_lr,
#                                   entropy_term.detach().cpu().item(), 
#                                   peakiness_term.detach().cpu().item(),
#                                   continuous_loss, discrete_loss]) 
        
#         # Concatenate the arrays
#         row = np.concatenate((prepend_array, max_values_array, token_ids_array))
#         new_row = pd.Series(row, index=df.columns)
#         df = pd.concat([df, new_row.to_frame().T], ignore_index=True)
#         # Save log data to CSV file periodically
#         if epoch_no % 10 == 0:
#             df.to_csv(csv_filename, index=False)
#         # Something went wrong; so break it and go to the next behavior.
#         if BREAK_IT==True:
#             break
#     # End of optimizations        
#     # Write to CSV file
#     df.to_csv(csv_filename, index=False)
#     # Token and Output Generation Step
#     inputs_token_ids = one_hot_inputs.argmax(dim=1)
#     adv_token_ids = effective_adv_one_hot.argmax(dim=1)
#     adv_suffix_list = str(adv_token_ids.cpu().numpy().tolist())
#     adv_suffix_string = tokenizer.decode(adv_token_ids.cpu().numpy())
#     final_string_ids = torch.hstack([inputs_token_ids, adv_token_ids])
#     outputs = []
#     generation_loop = range(1)
#     import warnings
#     # Suppress specific warnings
#     warnings.filterwarnings("ignore", message=".*`do_sample` is set to `False`. However, `temperature` is set to.*")
#     warnings.filterwarnings("ignore", message=".*`do_sample` is set to `False`. However, `top_p` is set to.*")
    
#     ######## Discrete tokens for Output Generation ########
#     for loop in tqdm(generation_loop, desc="Generating outputs", leave=False):
#         generated_output = model.generate(final_string_ids.unsqueeze(0), 
#                                           max_length=num_tokens, 
#                                           pad_token_id=tokenizer.pad_token_id,
#                                           do_sample=False)
#         generated_output_string = tokenizer.decode(generated_output[0][:].cpu().numpy(), skip_special_tokens=True).strip()
#         outputs.append(generated_output_string)
    
#     iteration_result = {
#         "harmful-behaviour": user_prompt,
#         "target": target,
#         "suffix_token_ids": adv_suffix_list,
#         "suffix_string": adv_suffix_string,
#         "epoch_no": epoch_no,
#         "continuous_loss": continuous_loss,
#         "discrete_loss": discrete_loss,
#         "best_disc_loss":best_disc_loss,
#         "epch_at_best_disc_loss": best_loss_at_epoch,
#         "outputs": outputs
#     }
#     optimization_results.append(iteration_result)

#     # Update the JSON File
#     with open(json_file_path, "w") as f:
#         json.dump(
#             optimization_results,
#             f,
#             indent=4,  # Keep structure readable
#             ensure_ascii=False  # Do this to Handle non-ASCII chars properly; If there is any
#         )

#     ##########################################################################################
#     ## Create and Update a JSONL file (required for running the parse_results.ipynb script) ##
#     output_file = directories[1]+"/output_"+model_name+"("+str(len(harmful_behaviors))+"_behaviors)("+str(num_steps)+"_steps).jsonl"
#     from behavior import Behavior
#     behavior = Behavior(user_prompt, adv_suffix_string, generated_output_string, "", "")
    
#     with open(output_file, 'a') as f:
#         f.write(json.dumps(behavior.to_dict()) + '\n')
#     f.close()  
#     ## Create and Update a JSONL file (required for running the parse_results.ipynb script) ##
#     ########################################################################################## s

In [ ]:
# # After it's done, compute the similarity and distance between the one_hot_adv_Safe and one_hot_adv_Unsafe
# print(len(one_hot_adv_list))
# # Use the Last embeddings_adv
# for one_hot_adv in one_hot_adv_list:
#     embeddings_adv = (one_hot_adv @ embed_weights).unsqueeze(0)
#     final_prompt_embeds = torch.hstack([embeddings_user, embeddings_adv])
#     outputs = []
#     generation_loop = range(10)
#     for i in tqdm(generation_loop, desc="Generating outputs", leave=False):
#         generated_output =  model.generate(inputs_embeds=final_prompt_embeds, max_length=num_tokens).squeeze()
#         generated_output_string = tokenizer.decode(generated_output, skip_special_tokens=True)
#         print(f"#{i}\n",generated_output_string)
#         outputs.append(generated_output_string)